# Retrieval-Augmented Generation (RAG)

**RAG** combines information retrieval with text generation. Instead of relying only on what a language model learned during training, a RAG system first finds relevant information from a knowledge source and then gives that information to the model as context for its answer.

## How this notebook's RAG pipeline works

1. **Load and prepare source data:** fetch a web page, clean its text, and split it into smaller chunks.
2. **Index the chunks:** an embedding model converts each chunk into a vector, and Chroma stores the vectors alongside the original text.
3. **Retrieve relevant information:** embed a question and search Chroma for chunks with similar meaning.
4. **Generate an answer:** combine the retrieved chunks into a context string, include it with the question in a prompt, and ask the language model to answer from that context.

In short: **question → retrieve relevant chunks → provide them as context → generate an answer**. The embedding vectors help find relevant text; the language model receives the retrieved text, not the vectors themselves.

## 1. Load source documents

Fetch the Google blog page and load its content as LangChain `Document` objects. These are the source documents that the RAG pipeline will clean, split, embed, store, and retrieve.

In [7]:
import torch 
import sys 

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Pythonb version: {sys.version}")


if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"cuDNN version: {torch.backends.cudnn.version()}")
else:
    print("Using CPU for computations. CUDA is not available.")

PyTorch version: 2.14.1+cpu
CUDA available: False
Pythonb version: 3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)]
Using CPU for computations. CUDA is not available.


In [ ]:
import re 
from bs4 import BeautifulSoup
from langchain_community.document_loaders import RecursiveUrlLoader

def extract_text(html_content):
    soup = BeautifulSoup(html_content, 'lxml')

    # Remove script and style elements
    for tag in soup.select(
        "script, style, noscript, iframe, header, footer, nav, aside"
    ):
        tag.decompose()
        
    return re.sub(r'\s+', ' ', soup.get_text()).strip()

In [ ]:
docs_urls = [
    "https://docs.python.org/3/",
    "https://docs.pytorch.org/docs/stable/",
    "https://huggingface.co/docs",
    "https://fastapi.tiangolo.com/",
    "https://scikit-learn.org/stable/",
    "https://www.tensorflow.org/guide",
    "https://docs.langchain.com/",
    "https://docs.llamaindex.ai/",
    "https://docs.trychroma.com/",
    "https://faiss.ai/",
    "https://docs.pydantic.dev/latest/",
    "https://numpy.org/doc/stable/",
    "https://pandas.pydata.org/docs/",
    "https://www.crummy.com/software/BeautifulSoup/bs4/doc/",
    "https://docs.scrapy.org/en/latest/"
]

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from langchain_community.document_loaders import RecursiveUrlLoader

from warnings import filterwarnings

filterwarnings("ignore", category=UserWarning, module="bs4")
def load_site(url, max_depth=2):
    loader = RecursiveUrlLoader(
        url=url,
        max_depth=max_depth,
        extractor=extract_text,
        prevent_outside=True,
        continue_on_failure=True,
        timeout=15,
        use_async=True,
        encoding="utf-8",
    )
    return loader.load()

docs = []
errors = []
site_counts = {}


with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {executor.submit(load_site, url): url for url in docs_urls}

    for future in as_completed(futures):
        url = futures[future]
        try:
            site_docs = future.result()
            docs.extend(site_docs)
            site_counts[url] = len(site_docs)
            print(f"{url}:{len(site_docs)} documents loaded.")
        except Exception as e:
            errors.append((url, str(e)))
            # print(f"Error loading {url}: {e}")

print(f"\nTotal documents loaded: {len(docs)}")


## 2. Split documents into chunks

Configure a recursive splitter to break long documents into chunks of about 1,000 characters, with 200 characters of overlap. Overlap helps preserve context across chunk boundaries. This first split is an initial pass; the next cell cleans the source and splits it again.

In [ ]:

import re
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

def clean_scraped_text(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    lines = [
        re.sub(r"[ \t]+", " ", line).strip()
        for line in text.split("\n")
    ]

    return "\n".join(
        line for line in lines if line
    ).strip()

clean_docs = []

for doc in docs:
    content = clean_scraped_text(doc.page_content)

    if content:
        clean_docs.append(
            Document(
                page_content=content,
                metadata=doc.metadata.copy()
            )
        )

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
)

split_texts = text_splitter.split_documents(clean_docs)

print("Clean documents:", len(clean_docs))
print("Chunks:", len(split_texts))


In [ ]:

print("Chunks:", len(split_texts))
print("Sample:", split_texts[0].page_content)
print("Metadata:", split_texts[0].metadata)

## 3. Clean and re-split the source text

Remove stray slash characters and collapse repeated whitespace in each document. Then split the cleaned documents again so the chunks used by the rest of the pipeline contain the cleaned text.

In [ ]:
import re

def clean_scraped_text(text: str) -> str:
    # 1. Remove unwanted standalone slashes or stray marks (like /and or / )
    text = re.sub(r'\s+/\s*', ' ', text) 
    
    # 2. Collapse multiple spaces/newlines into clean spacing
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

# Apply cleaning to all documents before splitting
for doc in docs:
    doc.page_content = clean_scraped_text(doc.page_content)

# Now split your cleaned documents safely
split_texts = text_splitter.split_documents(docs)


## 4. Count the chunks

Display how many text chunks were created. This is a quick check that splitting produced data before you inspect or store it.

In [ ]:
len(split_texts)

In [ ]:
import statistics

sizes = [len(chunk.page_content) for chunk in split_texts]

print(f"Chunks: {len(split_texts):,}")
print(f"Source documents: {len(docs):,}")
print(f"Average chunk size: {statistics.mean(sizes):,.0f} characters")
print(f"Median chunk size: {statistics.median(sizes):,.0f} characters")
print(f"Largest chunk: {max(sizes):,} characters")

## 5. Inspect the chunk contents

Print each chunk with a numbered label and separator. Reviewing these examples helps you check whether the source was cleaned and split into useful pieces.

In [ ]:

for i, chunk in enumerate(split_texts[:5], start=1):
    print(f"\nChunk {i}")
    print("Source:", chunk.metadata.get("source"))
    print("Length:", len(chunk.page_content))
    print(chunk.page_content[:700])


## 6. Set up the embedding model

Load the smaller `all-MiniLM-L6-v2` model for faster CPU embedding. It produces 384-dimensional vectors instead of MPNet's 768, so this notebook uses a separate Chroma collection and must use this same model for indexing and queries. The optional Hugging Face token can raise download limits.

In [8]:
from getpass import getpass
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "sentence-transformers/all-MiniLM-L6-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"
hf_token = getpass("Hugging Face token (leave blank for public models): ").strip()
model_kwargs = {"device": device}
if hf_token:
    model_kwargs["token"] = hf_token

print(f"Embedding device: {device}")
encode_kwargs = {
    "normalize_embeddings": True, 
    "batch_size": 64 if device == "cuda" else 16}

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)


Embedding device: cpu


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:

import time

sample_texts = [
    doc.page_content for doc in split_texts[:16]
]

start = time.perf_counter()
sample_vectors = embeddings.embed_documents(sample_texts)

print("Sample texts:", len(sample_texts))
print("Embedding dimensions:", len(sample_vectors[0]))
print("Elapsed seconds:", round(time.perf_counter() - start, 2))


## 7. Set up the Chroma vector store

Create or open the `my_collection` collection in Chroma, persist it in `./chroma_persist`, and connect the embedding model. Chroma will use that model to embed text when documents are added and queries are searched.

In [9]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    collection_name="docs_rag_minilm_v1",
    embedding_function=embeddings,
    persist_directory="./chroma_persist",

)
print("Initial collection count:",
      vectorstore._collection.count())


Initial collection count: 12931


## 8. Add the chunks to Chroma

Store the cleaned chunks in the vector store. Chroma creates embeddings for the text and saves the document content and associated data; the returned IDs identify the stored records.

In [ ]:
import hashlib
import time 
from math import ceil

BATCH_SIZE = 256

def make_chunk_id(doc):
    source = doc.metadata.get("source", "")
    raw = f'{source}:{doc.page_content}'
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

def add_documents_in_batches(
    vectorstore, documents, batch_size=256
):
    total = len(documents)
    total_batches = ceil(total / batch_size)
    start = time.perf_counter()
    seen_ids = set()

    for i in range(0, total, batch_size):
        batch = documents[i:i + batch_size]
        ids = []

        for doc in batch:
            base_id = make_chunk_id(doc)
            chunk_id = base_id
            duplicate_index = 1

            while chunk_id in seen_ids:
                chunk_id = hashlib.sha256(
                    f"{base_id}:{duplicate_index}".encode("utf-8")
                ).hexdigest()
                duplicate_index += 1

            seen_ids.add(chunk_id)
            ids.append(chunk_id)

        batch_start = time.perf_counter()

        vectorstore.add_documents(
            documents=batch,
            ids=ids,
        )

        elapsed = time.perf_counter() - batch_start
        batch_num = i // batch_size + 1

        print(
            f"Batch {batch_num}/{total_batches} | "
            f"{min(i + len(batch), total)}/{total} chunks | "
            f"{elapsed:.2f}s"
        )

    print(
        "Total time:",
        round(time.perf_counter() - start, 2),
        "seconds"
    )
    print(
        "Stored records:",
        vectorstore._collection.count()
    )

add_documents_in_batches(
    vectorstore,
    split_texts,
    BATCH_SIZE
)


## 9. Inspect the stored collection

Fetch and print the collection's stored records, then print the number of records. This is a basic check that the chunks were added to Chroma.

In [10]:

print("Total stored chunks:",
      vectorstore._collection.count())

sample = vectorstore._collection.get(
    limit=3,
    include=["documents", "metadatas"]
)

for i, doc_id in enumerate(sample["ids"]):
    print(f"\nRecord {i + 1}")
    print("ID:", doc_id)
    print("Source:",
          sample["metadatas"][i].get("source"))
    print("Text:",
          sample["documents"][i][:300])


Total stored chunks: 12931

Record 1
ID: f2ad6555b88c5992647de3efb06cc03d17db4bcdbaad417f0ed95fd5bcb47e32
Source: https://docs.pytorch.org/docs/stable/
Text: Redirecting… Continue to ../2.14/index.html

Record 2
ID: 6b9830c4b24fd25c70dd23574fefa67b6764fa14e9d9df3404e04bd3a2cc307b
Source: https://docs.pytorch.org/docs/2.14/index.html
Text: PyTorch documentation — PyTorch 2.14 documentation Skip to main content Back to top Ctrl+K Go to pytorch.org Ctrl+K X GitHub PyTorch Forum PyPi PyTorch documentation# PyTorch is an optimized tensor library for deep learning using GPUs and CPUs. Features described in this documentation are classified

Record 3
ID: ca74fac2cbd87ddd22ee83baa841861f6f5b0ba9483fbd362b31136f1c46d52f
Source: https://docs.pytorch.org/docs/2.14/index.html
Text: . Install PyTorch User Guide PyTorch Overview Get Started Learn the Basics PyTorch Main Components Torch.compile Torch.export Developer Notes Accelerator Integration Reference API C++ torch torch.nn torch.nn.functional 

## 10. Optional: inspect one stored record

The commented example shows how to fetch a specific Chroma record by ID, including its embedding and document text. Uncomment it and use a valid ID if you want to inspect one item in more detail.

In [ ]:
# print("\n Collection 1 - ", vectorstore._collection.get(ids=['563458d8-5f27-4786-9a35-bd0e96bd06c5'], include=['embeddings', 'documents']))

## 11. Try a similarity search

Search Chroma for the three chunks most similar to the example query, `Gemini 4 Argon`, and print the returned documents. The query is embedded with the same embedding model before matching.

In [ ]:
# Augumentation of the documents in the vectorstore

retrieved_docs = vectorstore.similarity_search("Python", k=3)
print("Retrieved Documents:", retrieved_docs)

## 12. Configure a reusable retriever

Wrap the vector store as a retriever configured to return three results. A retriever provides a standard interface for fetching relevant documents; the final cell below performs its search directly on `vectorstore`.

In [ ]:
retreiver = vectorstore.as_retriever(
    search_type='mmr',  # or 'similarity' depending on your use case
    search_kwargs={
        "k": 3,
        "fetch_k": 20, 
        "lambda_mult": 0.7
     } # Number of documents to fetch before re-ranking}
    )

results = retreiver.invoke("How does python handle exceptions?")

for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content[:500])


## 13. Set up the chat model

Configure Ollama to run the local `llama3.1:8b` model. The temperature and other options control generation behavior; Ollama must be installed and the model available locally.

In [ ]:
!ollama pull qwen2.5:3b

In [19]:
from langchain_ollama import ChatOllama


llm = ChatOllama(model="qwen2.5:3b", temperature=0.1,top_p=0.9, stop=["\n\n"])

## 14. Retrieve context and generate an answer

Build a prompt that asks the model to answer only from the retrieved context. For the question, retrieve the four most similar documents, join their `page_content` into one context string, and pass both values through the prompt, chat model, and string output parser.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


retreiver = vectorstore.as_retriever(
    search_type='mmr',  # or 'similarity' depending on your use case
    search_kwargs={
        "k": 3,
        # "fetch_k": 20, 
        "lambda_mult": 0.7
     } # Number of documents to fetch before re-ranking}
    )

prompt = ChatPromptTemplate.from_template("""
Answer the question using the supplied documentation.

If the context does not contain enough evidence, say you
do not have enough information to answer.

Do not invent facts or claim the sources say something
that they do not support.
Give Citations in the form of [source](url) for any information you use from the context.If you use multiple sources, give citations for each source you use.

Context:
{context}

Question:
{question}

Answer:
""")

rag_chain = prompt | llm | StrOutputParser()

question = "Tell me about Python and FastAPI and detail about the different access options"
docs = vectorstore.similarity_search(question, k=4)
retrieved_docs = retreiver.invoke(question)
context = "\n\n".join(
    f"Source: {doc.metadata.get('source')}\n{doc.page_content}\n" 
    for doc in retrieved_docs
    )

answer = rag_chain.invoke({
    "context": context,
    "question": question,
})

print("Answer\n", answer, end="\n\n")
print("\nRetrieved Documents:")

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n{i}. {doc.metadata.get('source', 'Unknown')}")
    print(doc.page_content[:400])

Answer
 Python is a high-level, general-purpose programming language that supports multiple paradigms, including object-oriented, imperative, and functional programming. It is known for its readability and ease of use, which makes it popular for web development, data analysis, artificial intelligence, and more.


Retrieved Documents:

1. https://fastapi.tiangolo.com/
FastAPI - FastAPI Skip to content FastAPI Conf '26 October 28, 2026 Amsterdam, NL All about FastAPI, right from the source. Learn more → FastAPI Conf '26 October 28, 2026 Amsterdam, NL All about FastAPI, right from the source. Learn more → FastAPI¶ FastAPI framework, high performance, easy to learn, fast to code, ready for production Documentation: https://fastapi.tiangolo.com Source Code: https:/

2. https://fastapi.tiangolo.com/tutorial/security/oauth2-jwt/
. In almost any framework handling the security becomes a rather complex subject quite quickly. Many packages that simplify it a lot have to make many compromises wit